<!-- estilo-montserrat -->
<style>
  * { font-family: 'Montserrat', sans-serif !important; }
  h1, h2, h3, h4, h5, h6 { font-weight: 900 !important; }
  h1 { color: #4f8ef7; border-bottom: 2px solid #4f8ef7; padding-bottom: 4px; }
  h2 { color: #a66cff; border-bottom: 1px solid #a66cff; padding-bottom: 3px; }
  h3 { color: #ff6b9d; }
  h4, h5 { color: #ffa94d; }
  strong, b { color: #2ec4b6; }
  em, i { color: #f4c542; }
  li::marker { color: #ff6b9d; }
  blockquote { border-left: 4px solid #a66cff; background: rgba(166,108,255,0.10); padding: 4px 12px; margin-left: 0; }
  code { color: #ff7eb6; background: rgba(255,126,182,0.12); padding: 1px 5px; border-radius: 4px; }
  pre { background: rgba(79,142,247,0.10); border-left: 4px solid #4f8ef7; border-radius: 6px; padding: 8px 12px; }
  pre code { background: none; padding: 0; }
  a { color: #2ec4b6; }
  pre code { color: inherit; }
  pre.sql code .kw { color: #ff6b9d; font-weight: 700; }
  pre.sql code .fn { color: #4f8ef7; font-weight: 600; }
  pre.sql code .ar { color: #7ee787; }
  pre.sql code .ty { color: #2ec4b6; }
  pre.sql code .va { color: #c792ea; }
  pre.sql code .st { color: #f4c542; }
  pre.sql code .nu { color: #ffa94d; }
  pre.sql code .cm { color: #8b949e; font-style: italic; }
</style>

# Cursores
Un cursor recorre el resultado de un `SELECT` **fila por fila**, guardando los valores de cada fila en variables. Se usa cuando hay que hacer algo distinto por cada fila y no alcanza con una sola sentencia.

**Ciclo de vida (5 pasos):**
1. `DECLARE`: defino el nombre del cursor y le asigno el `SELECT`
2. `OPEN`: se ejecuta la consulta y se llena el cursor con los datos en memoria
3. `FETCH NEXT`: saco la fila actual, cargo sus valores en variables y el puntero avanza a la siguiente
4. `CLOSE`: cierro el cursor (libera el resultado, pero la definición sigue existiendo)
5. `DEALLOCATE`: destruyo la definición y libero toda la memoria

`@@FETCH_STATUS` dice cómo salió el último `FETCH`: `0` = se obtuvo una fila, `-1` = falló o ya no hay más filas, `-2` = la fila ya no existe.

> **Importante:** hay un `FETCH` antes del `WHILE` y otro **al final** del bucle. Si falta el segundo, el cursor nunca avanza y el bucle es infinito.

In [ ]:
DECLARE @clie CHAR(6)
DECLARE @detalle CHAR(100)

DECLARE mi_cursor CURSOR FOR                              -- 1) declaro
    SELECT clie_codigo, clie_razon_social
    FROM Cliente
    WHERE clie_codigo BETWEEN '00000' AND '00003'

OPEN mi_cursor                                            -- 2) abro

FETCH NEXT FROM mi_cursor INTO @clie, @detalle            -- 3) primera fila

WHILE @@FETCH_STATUS = 0                                  -- mientras la ultima lectura haya traido una fila
BEGIN
    PRINT 'Codigo: ' + @clie + ' Razon Social: ' + @detalle
    FETCH NEXT FROM mi_cursor INTO @clie, @detalle        -- siguiente fila (si no, bucle infinito)
END

CLOSE mi_cursor                                           -- 4) cierro
DEALLOCATE mi_cursor                                      -- 5) libero

<!-- estilo-montserrat -->
<style>
  * { font-family: 'Montserrat', sans-serif !important; }
  h1, h2, h3, h4, h5, h6 { font-weight: 900 !important; }
  h1 { color: #4f8ef7; border-bottom: 2px solid #4f8ef7; padding-bottom: 4px; }
  h2 { color: #a66cff; border-bottom: 1px solid #a66cff; padding-bottom: 3px; }
  h3 { color: #ff6b9d; }
  h4, h5 { color: #ffa94d; }
  strong, b { color: #2ec4b6; }
  em, i { color: #f4c542; }
  li::marker { color: #ff6b9d; }
  blockquote { border-left: 4px solid #a66cff; background: rgba(166,108,255,0.10); padding: 4px 12px; margin-left: 0; }
  code { color: #ff7eb6; background: rgba(255,126,182,0.12); padding: 1px 5px; border-radius: 4px; }
  pre { background: rgba(79,142,247,0.10); border-left: 4px solid #4f8ef7; border-radius: 6px; padding: 8px 12px; }
  pre code { background: none; padding: 0; }
  a { color: #2ec4b6; }
  pre code { color: inherit; }
  pre.sql code .kw { color: #ff6b9d; font-weight: 700; }
  pre.sql code .fn { color: #4f8ef7; font-weight: 600; }
  pre.sql code .ar { color: #7ee787; }
  pre.sql code .ty { color: #2ec4b6; }
  pre.sql code .va { color: #c792ea; }
  pre.sql code .st { color: #f4c542; }
  pre.sql code .nu { color: #ffa94d; }
  pre.sql code .cm { color: #8b949e; font-style: italic; }
</style>

### Cursor para modificar filas
Con `FOR UPDATE OF campo` el cursor permite modificar, y `WHERE CURRENT OF nombre_cursor` hace el `UPDATE` sobre la fila en la que está parado.

In [ ]:
DECLARE @clie CHAR(6)
DECLARE @detalle CHAR(100)

DECLARE mi_cursor CURSOR FOR
    SELECT clie_codigo, clie_razon_social
    FROM Cliente
    WHERE clie_codigo BETWEEN '00000' AND '00003'
    FOR UPDATE OF clie_razon_social

OPEN mi_cursor
FETCH NEXT FROM mi_cursor INTO @clie, @detalle

WHILE @@FETCH_STATUS = 0
BEGIN
    IF @clie = '00001'
    BEGIN
        UPDATE Cliente
        SET clie_razon_social = 'NUEVA RAZON SOCIAL'
        WHERE CURRENT OF mi_cursor                        -- actualiza solo la fila actual del cursor
    END

    PRINT 'Codigo: ' + @clie + ' Razon Social: ' + @detalle
    FETCH NEXT FROM mi_cursor INTO @clie, @detalle
END

CLOSE mi_cursor
DEALLOCATE mi_cursor

<!-- estilo-montserrat -->
<style>
  * { font-family: 'Montserrat', sans-serif !important; }
  h1, h2, h3, h4, h5, h6 { font-weight: 900 !important; }
  h1 { color: #4f8ef7; border-bottom: 2px solid #4f8ef7; padding-bottom: 4px; }
  h2 { color: #a66cff; border-bottom: 1px solid #a66cff; padding-bottom: 3px; }
  h3 { color: #ff6b9d; }
  h4, h5 { color: #ffa94d; }
  strong, b { color: #2ec4b6; }
  em, i { color: #f4c542; }
  li::marker { color: #ff6b9d; }
  blockquote { border-left: 4px solid #a66cff; background: rgba(166,108,255,0.10); padding: 4px 12px; margin-left: 0; }
  code { color: #ff7eb6; background: rgba(255,126,182,0.12); padding: 1px 5px; border-radius: 4px; }
  pre { background: rgba(79,142,247,0.10); border-left: 4px solid #4f8ef7; border-radius: 6px; padding: 8px 12px; }
  pre code { background: none; padding: 0; }
  a { color: #2ec4b6; }
  pre code { color: inherit; }
  pre.sql code .kw { color: #ff6b9d; font-weight: 700; }
  pre.sql code .fn { color: #4f8ef7; font-weight: 600; }
  pre.sql code .ar { color: #7ee787; }
  pre.sql code .ty { color: #2ec4b6; }
  pre.sql code .va { color: #c792ea; }
  pre.sql code .st { color: #f4c542; }
  pre.sql code .nu { color: #ffa94d; }
  pre.sql code .cm { color: #8b949e; font-style: italic; }
</style>

# Transacciones y niveles de aislamiento
Una **transacción** agrupa sentencias que se confirman o se deshacen juntas.
- `BEGIN TRANSACTION`: la abre. `@@TRANCOUNT` dice cuántas hay abiertas.
- `COMMIT`: confirma los cambios.
- `ROLLBACK`: los deshace.

Mientras una transacción está abierta, las filas que tocó quedan **bloqueadas**. El **nivel de aislamiento** define qué tanto respeta cada transacción los bloqueos de las otras y qué problemas puede ver:

| Problema | READ UNCOMMITTED | READ COMMITTED | REPEATABLE READ | SERIALIZABLE |
|---|:---:|:---:|:---:|:---:|
| **Dato sucio** (leer algo no confirmado) | Sí | No | No | No |
| **Lectura no repetible** (releo la fila y cambió) | Sí | Sí | No | No |
| **Dato fantasma** (releo el rango y aparecen filas nuevas) | Sí | Sí | Sí | No |

Más aislamiento = menos problemas pero más bloqueos. `READ COMMITTED` es el default de SQL Server.

### Cómo probarlo
Estos ejemplos se corren en **dos conexiones/ventanas distintas a la vez** (Sesión A y Sesión B), no en el mismo cuaderno, porque lo que se prueba es cómo se pisan entre sí. Si las corrés en el mismo cuaderno una detrás de la otra no se nota nada.

<!-- estilo-montserrat -->
<style>
  * { font-family: 'Montserrat', sans-serif !important; }
  h1, h2, h3, h4, h5, h6 { font-weight: 900 !important; }
  h1 { color: #4f8ef7; border-bottom: 2px solid #4f8ef7; padding-bottom: 4px; }
  h2 { color: #a66cff; border-bottom: 1px solid #a66cff; padding-bottom: 3px; }
  h3 { color: #ff6b9d; }
  h4, h5 { color: #ffa94d; }
  strong, b { color: #2ec4b6; }
  em, i { color: #f4c542; }
  li::marker { color: #ff6b9d; }
  blockquote { border-left: 4px solid #a66cff; background: rgba(166,108,255,0.10); padding: 4px 12px; margin-left: 0; }
  code { color: #ff7eb6; background: rgba(255,126,182,0.12); padding: 1px 5px; border-radius: 4px; }
  pre { background: rgba(79,142,247,0.10); border-left: 4px solid #4f8ef7; border-radius: 6px; padding: 8px 12px; }
  pre code { background: none; padding: 0; }
  a { color: #2ec4b6; }
  pre code { color: inherit; }
  pre.sql code .kw { color: #ff6b9d; font-weight: 700; }
  pre.sql code .fn { color: #4f8ef7; font-weight: 600; }
  pre.sql code .ar { color: #7ee787; }
  pre.sql code .ty { color: #2ec4b6; }
  pre.sql code .va { color: #c792ea; }
  pre.sql code .st { color: #f4c542; }
  pre.sql code .nu { color: #ffa94d; }
  pre.sql code .cm { color: #8b949e; font-style: italic; }
</style>

#### 1. Dato sucio
La Sesión A modifica sin confirmar. La Sesión B lee la misma fila:
- con `READ UNCOMMITTED` **ve el dato sucio** (`X10`) porque ignora los bloqueos
- con el nivel por defecto (`READ COMMITTED`) **se queda esperando** hasta que A haga `COMMIT` o `ROLLBACK`

In [ ]:
-- SESION A (se ejecuta primero)
BEGIN TRANSACTION
    SELECT @@TRANCOUNT                                    -- 1: hay una transaccion abierta
    UPDATE Cliente SET clie_razon_social = 'X10' WHERE clie_codigo = '00000'
    -- hasta aca la transaccion queda abierta (fila bloqueada)
ROLLBACK                                                  -- la deshace (con COMMIT la confirmaria)

In [ ]:
-- SESION B (se ejecuta mientras A sigue abierta)
BEGIN TRANSACTION
    SET TRANSACTION ISOLATION LEVEL READ UNCOMMITTED
    SELECT * FROM Cliente WHERE clie_codigo = '00000'     -- ve 'X10', un dato que despues A puede deshacer
COMMIT

<!-- estilo-montserrat -->
<style>
  * { font-family: 'Montserrat', sans-serif !important; }
  h1, h2, h3, h4, h5, h6 { font-weight: 900 !important; }
  h1 { color: #4f8ef7; border-bottom: 2px solid #4f8ef7; padding-bottom: 4px; }
  h2 { color: #a66cff; border-bottom: 1px solid #a66cff; padding-bottom: 3px; }
  h3 { color: #ff6b9d; }
  h4, h5 { color: #ffa94d; }
  strong, b { color: #2ec4b6; }
  em, i { color: #f4c542; }
  li::marker { color: #ff6b9d; }
  blockquote { border-left: 4px solid #a66cff; background: rgba(166,108,255,0.10); padding: 4px 12px; margin-left: 0; }
  code { color: #ff7eb6; background: rgba(255,126,182,0.12); padding: 1px 5px; border-radius: 4px; }
  pre { background: rgba(79,142,247,0.10); border-left: 4px solid #4f8ef7; border-radius: 6px; padding: 8px 12px; }
  pre code { background: none; padding: 0; }
  a { color: #2ec4b6; }
  pre code { color: inherit; }
  pre.sql code .kw { color: #ff6b9d; font-weight: 700; }
  pre.sql code .fn { color: #4f8ef7; font-weight: 600; }
  pre.sql code .ar { color: #7ee787; }
  pre.sql code .ty { color: #2ec4b6; }
  pre.sql code .va { color: #c792ea; }
  pre.sql code .st { color: #f4c542; }
  pre.sql code .nu { color: #ffa94d; }
  pre.sql code .cm { color: #8b949e; font-style: italic; }
</style>

#### 2. Lectura no repetible
Con `REPEATABLE READ`, la Sesión A mantiene bloqueadas las filas que leyó hasta terminar. Si la Sesión B intenta modificarlas, **queda esperando** hasta que A haga `COMMIT`. (Con `READ COMMITTED` B podría modificarla y A, al volver a leer, vería otro valor.)

In [ ]:
-- SESION A
BEGIN TRANSACTION
    SET TRANSACTION ISOLATION LEVEL REPEATABLE READ

    SELECT * FROM Cliente WHERE clie_codigo = '00000'
    SELECT * FROM Cliente WHERE clie_codigo = '00001'
COMMIT

In [ ]:
-- SESION B (mientras A no hizo COMMIT)
BEGIN TRANSACTION
    UPDATE Cliente SET clie_razon_social = 'X1' WHERE clie_codigo = '00000'    -- se queda esperando hasta que A confirme
COMMIT

<!-- estilo-montserrat -->
<style>
  * { font-family: 'Montserrat', sans-serif !important; }
  h1, h2, h3, h4, h5, h6 { font-weight: 900 !important; }
  h1 { color: #4f8ef7; border-bottom: 2px solid #4f8ef7; padding-bottom: 4px; }
  h2 { color: #a66cff; border-bottom: 1px solid #a66cff; padding-bottom: 3px; }
  h3 { color: #ff6b9d; }
  h4, h5 { color: #ffa94d; }
  strong, b { color: #2ec4b6; }
  em, i { color: #f4c542; }
  li::marker { color: #ff6b9d; }
  blockquote { border-left: 4px solid #a66cff; background: rgba(166,108,255,0.10); padding: 4px 12px; margin-left: 0; }
  code { color: #ff7eb6; background: rgba(255,126,182,0.12); padding: 1px 5px; border-radius: 4px; }
  pre { background: rgba(79,142,247,0.10); border-left: 4px solid #4f8ef7; border-radius: 6px; padding: 8px 12px; }
  pre code { background: none; padding: 0; }
  a { color: #2ec4b6; }
  pre code { color: inherit; }
  pre.sql code .kw { color: #ff6b9d; font-weight: 700; }
  pre.sql code .fn { color: #4f8ef7; font-weight: 600; }
  pre.sql code .ar { color: #7ee787; }
  pre.sql code .ty { color: #2ec4b6; }
  pre.sql code .va { color: #c792ea; }
  pre.sql code .st { color: #f4c542; }
  pre.sql code .nu { color: #ffa94d; }
  pre.sql code .cm { color: #8b949e; font-style: italic; }
</style>

#### 3. Deadlock (bloqueo mutuo)
Las **dos** sesiones corren el mismo bloque con `REPEATABLE READ`: leen la fila y después intentan modificarla. Cada una tiene un bloqueo de lectura sobre la fila, así que la modificación de una espera a que la otra suelte el suyo y ninguna puede avanzar. SQL Server lo detecta y **cancela una de las dos** (la "víctima"). Orden para provocarlo: A lee → B lee → A modifica (espera) → B modifica (deadlock).

In [ ]:
-- SESION A y SESION B corren el mismo bloque
BEGIN TRANSACTION
    SET TRANSACTION ISOLATION LEVEL REPEATABLE READ

    SELECT * FROM Cliente WHERE clie_codigo = '00000'
    UPDATE Cliente SET clie_razon_social = 'X1' WHERE clie_codigo = '00000'
COMMIT

ROLLBACK       -- si una transaccion quedo abierta o fue la victima, la cierro

<!-- estilo-montserrat -->
<style>
  * { font-family: 'Montserrat', sans-serif !important; }
  h1, h2, h3, h4, h5, h6 { font-weight: 900 !important; }
  h1 { color: #4f8ef7; border-bottom: 2px solid #4f8ef7; padding-bottom: 4px; }
  h2 { color: #a66cff; border-bottom: 1px solid #a66cff; padding-bottom: 3px; }
  h3 { color: #ff6b9d; }
  h4, h5 { color: #ffa94d; }
  strong, b { color: #2ec4b6; }
  em, i { color: #f4c542; }
  li::marker { color: #ff6b9d; }
  blockquote { border-left: 4px solid #a66cff; background: rgba(166,108,255,0.10); padding: 4px 12px; margin-left: 0; }
  code { color: #ff7eb6; background: rgba(255,126,182,0.12); padding: 1px 5px; border-radius: 4px; }
  pre { background: rgba(79,142,247,0.10); border-left: 4px solid #4f8ef7; border-radius: 6px; padding: 8px 12px; }
  pre code { background: none; padding: 0; }
  a { color: #2ec4b6; }
  pre code { color: inherit; }
  pre.sql code .kw { color: #ff6b9d; font-weight: 700; }
  pre.sql code .fn { color: #4f8ef7; font-weight: 600; }
  pre.sql code .ar { color: #7ee787; }
  pre.sql code .ty { color: #2ec4b6; }
  pre.sql code .va { color: #c792ea; }
  pre.sql code .st { color: #f4c542; }
  pre.sql code .nu { color: #ffa94d; }
  pre.sql code .cm { color: #8b949e; font-style: italic; }
</style>

## Ejercicios para practicar
Ninguno de los ejercicios de la práctica necesita cursores ni transacciones, así que quedan estos de complejidad similar a los de la clase. Los que modifican datos, hacelos sobre una **copia** (`SELECT * INTO mi_cliente FROM Cliente`) o dentro de `BEGIN TRANSACTION` + `ROLLBACK`, para no tocar la tabla real:

1. **Cursor de solo lectura:** recorrer los clientes con límite de crédito >= $1000 e imprimir, por cada uno, razón social y cantidad de facturas. Al final imprimir cuántos clientes se procesaron.
2. **Cursor con corte de control:** recorrer las facturas ordenadas por año y vendedor e imprimir una línea por vendedor y año con su monto total, y un subtotal por año.
3. **Cursor para modificar:** sobre `mi_cliente`, con `FOR UPDATE OF` y `WHERE CURRENT OF`, subir un 10% el límite de crédito de los clientes que facturaron en 2012. Verificar el resultado contra `Cliente`.
4. **Transacción:** abrir una transacción, modificar varias filas de `mi_cliente`, mostrar `@@TRANCOUNT`, hacer `ROLLBACK` y comprobar que los datos volvieron. Repetir con `COMMIT`.
5. **Aislamiento (en dos pestañas):** escribir primero qué esperás que pase y después comprobarlo: (a) lectura no repetible con `READ COMMITTED` vs. `REPEATABLE READ`; (b) un deadlock cruzando el orden de dos filas (A modifica la fila 1 y luego la 2; B la 2 y luego la 1).